# Demo 10c — DBSCAN, density-based clustering

**Class 10 · C1** (slides: `slides/10_clustering.md`)

A point is a **core** point when at least `min_samples` points (itself included) lie within distance $\varepsilon$;
a **border** point is within $\varepsilon$ of a core point but is not core; the rest is **noise** (label -1).

* Core, border and noise on a small example.
* Sweep of $\varepsilon$ and `min_samples` on the moons.
* The $k$-distance graph, and the knee found by `kneeliverse`.
* Failure cases: varying densities and high dimension.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from kneeliverse import kneedle, menger
from sklearn.cluster import DBSCAN, KMeans
from sklearn.metrics import adjusted_rand_score as ari
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

from faa import clustering as cl
from faa import demo, plotlib
from faa.demo import show, slide

demo.setup("10")
BLUE, ORANGE, GREEN, RED, SKY, PURPLE, GREY = (
    plotlib.BLUE,
    plotlib.ORANGE,
    plotlib.GREEN,
    plotlib.RED,
    plotlib.SKY,
    plotlib.PURPLE,
    plotlib.GREY,
)
COLS = [BLUE, ORANGE, GREEN, RED, SKY, PURPLE]
SEED = 0
B = cl.benchmarks(SEED)

## Slide: Core, border and noise

Twelve points, $\varepsilon = 1.2$, `min_samples` = 4. The count includes the point itself.

In [ ]:
P = np.array(
    [
        [1.0, 1.0],
        [1.6, 1.4],
        [2.0, 1.0],
        [1.5, 2.0],
        [2.4, 1.8],
        [3.3, 1.9],
        [7.0, 6.0],
        [7.5, 6.6],
        [8.1, 6.2],
        [7.4, 5.2],
        [8.6, 7.0],
        [4.5, 8.5],
    ]
)
EPS, MP = 1.2, 4
db = DBSCAN(eps=EPS, min_samples=MP).fit(P)
core = np.zeros(len(P), bool)
core[db.core_sample_indices_] = True
kind = np.where(core, "core", np.where(db.labels_ >= 0, "border", "noise"))
counts = (np.linalg.norm(P[:, None] - P[None], axis=-1) <= EPS).sum(1)
slide("Core, border and noise")
show(pl.DataFrame({"point": range(len(P)), "neighbours in eps": counts, "type": kind, "cluster": db.labels_}))
f, ax = plt.subplots(figsize=(plotlib.COL, 1.7))
mark = {"core": "o", "border": "s", "noise": "x"}
for t in ("core", "border", "noise"):
    m = kind == t
    ax.scatter(*P[m].T, marker=mark[t], c=[GREY if v < 0 else COLS[v] for v in db.labels_[m]], s=26, label=t)
for p in P[core]:
    ax.add_patch(plt.Circle(p, EPS, fill=False, lw=0.3, color=GREY))
ax.set_aspect("equal"), ax.legend(loc="lower right")
plotlib.save(f, "c1_core_border_noise")

## Slide: Two parameters decide the answer

The moons ($n = 300$, noise 0.07), `min_samples` = 5 and a sweep of $\varepsilon$. Too small: everything is noise
or many fragments; too large: the two moons merge into one.

In [ ]:
Xm, ym = B["moons"]
rows = []
for eps in (0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.6, 0.8):
    lab = DBSCAN(eps=eps, min_samples=5).fit_predict(Xm)
    rows.append((eps, len(set(lab)) - (-1 in lab), int((lab == -1).sum()), ari(ym, lab)))
sweep = pl.DataFrame(rows, schema=["eps", "clusters", "noise points", "ARI"], orient="row")
slide("Two parameters decide the answer")
show(sweep)
f, ax = plt.subplots(1, 4, figsize=(plotlib.FULL, 1.25), sharex=True, sharey=True)
for a, eps in zip(ax, (0.05, 0.2, 0.4, 0.8)):
    lab = DBSCAN(eps=eps, min_samples=5).fit_predict(Xm)
    a.scatter(*Xm.T, c=[GREY if v < 0 else COLS[v % 6] for v in lab], s=4, lw=0)
    a.set_title(f"$\\varepsilon$ = {eps}"), a.set_xticks([]), a.set_yticks([])
plotlib.save(f, "c1_eps_sweep")

## Slide: The $k$-distance graph

For `min_samples` = $m$, sort the distance of every point to its $m$-th neighbour (itself counts as the first).
Points inside a cluster have small distances, noise has large ones: the knee is a good $\varepsilon$.
`kneeliverse` finds the knee without looking at the plot.

In [ ]:
def kdist(X, m):
    d, _ = NearestNeighbors(n_neighbors=m).fit(X).kneighbors(X)
    return np.sort(d[:, -1])


rows = []
f, ax = plt.subplots(1, 2, figsize=(plotlib.FULL, 1.6))
for name, m, a in (("moons", 5, ax[0]), ("blobs", 5, ax[1])):
    Xk, yk = B[name]
    d = kdist(Xk, m)
    pts = np.c_[np.arange(len(d)), d]
    i_kd, i_mg = kneedle.knee(pts), menger.knee(pts)
    lab = DBSCAN(eps=float(d[i_kd]), min_samples=m).fit_predict(Xk)
    rows.append(
        (name, m, float(d[i_kd]), float(d[i_mg]), len(set(lab)) - (-1 in lab), int((lab == -1).sum()), ari(yk, lab))
    )
    a.plot(d, color=BLUE)
    a.axhline(d[i_kd], color=RED, ls="--", lw=0.7)
    a.scatter([i_kd], [d[i_kd]], color=RED, zorder=3)
    (
        a.set_title(f"{name}: $\\varepsilon$ = {d[i_kd]:.2f}"),
        a.set_xlabel("points sorted by distance"),
        a.set_ylabel(f"distance to neighbour {m}"),
    )
plotlib.save(f, "c1_kdistance")
slide("The k-distance graph")
show(
    pl.DataFrame(
        rows,
        schema=["set", "min_samples", "eps (Kneedle)", "eps (Menger)", "clusters", "noise points", "ARI"],
        orient="row",
    )
)

## Slide: `min_samples` and the noise it declares

A larger `min_samples` demands more neighbours: more points become noise, small clusters disappear. Rule of
thumb: `min_samples` $\geq d + 1$, and $2d$ for noisy data. A sweep on the moons with a noisier copy (std 0.15).

In [ ]:
from sklearn.datasets import make_moons

Xn, yn = make_moons(300, noise=0.15, random_state=SEED)
rows = []
for m in (3, 5, 10, 20, 40):
    d = kdist(Xn, m)
    i = kneedle.knee(np.c_[np.arange(len(d)), d])
    lab = DBSCAN(eps=float(d[i]), min_samples=m).fit_predict(Xn)
    rows.append((m, float(d[i]), len(set(lab)) - (-1 in lab), int((lab == -1).sum()), ari(yn, lab)))
slide("min_samples and the noise it declares")
show(pl.DataFrame(rows, schema=["min_samples", "eps (knee)", "clusters", "noise points", "ARI"], orient="row"))

## Failure 1: varying densities

One $\varepsilon$ cannot serve a dense and a sparse cluster. The `varied` set (std 0.4, 1.6, 0.8): small
$\varepsilon$ turns the sparse cluster into noise, large $\varepsilon$ merges the dense ones.

In [ ]:
Xv, yv = B["varied"]
rows = []
for eps in (0.25, 0.4, 0.6, 0.9, 1.3):
    lab = DBSCAN(eps=eps, min_samples=5).fit_predict(Xv)
    rows.append((eps, len(set(lab)) - (-1 in lab), int((lab == -1).sum()), ari(yv, lab)))
slide("Varying densities")
show(pl.DataFrame(rows, schema=["eps", "clusters", "noise points", "ARI"], orient="row"))
print(f"k-means with k = 3 on the same set: ARI {ari(yv, KMeans(3, n_init=10, random_state=SEED).fit_predict(Xv)):.3f}")
f, ax = plt.subplots(1, 3, figsize=(plotlib.FULL, 1.3), sharex=True, sharey=True)
for a, eps in zip(ax, (0.4, 0.6, 1.3)):
    lab = DBSCAN(eps=eps, min_samples=5).fit_predict(Xv)
    a.scatter(*Xv.T, c=[GREY if v < 0 else COLS[v % 6] for v in lab], s=4, lw=0)
    a.set_title(f"$\\varepsilon$ = {eps}"), a.set_xticks([]), a.set_yticks([])
plotlib.save(f, "c1_varied")

## Failure 2: distances lose meaning in high dimension

300 uniform points in $d$ dimensions. The ratio between the farthest and the nearest point tends to 1 as $d$
grows, so a single $\varepsilon$ separates less and less. Digits (64 pixels) are the real case.

In [ ]:
rows = []
for d in (2, 10, 50, 200, 1000):
    rng = np.random.default_rng(SEED)
    Xd = rng.uniform(0, 1, (300, d))
    D = np.linalg.norm(Xd[:, None] - Xd[None], axis=-1)
    np.fill_diagonal(D, np.inf)
    nn = D.min(1)
    far = np.where(np.isinf(D), -np.inf, D).max(1)
    rows.append((d, float((far / nn).mean())))
slide("Distances in high dimension")
show(pl.DataFrame(rows, schema=["d", "farthest / nearest neighbour"], orient="row"))
Xg, yg, _ = __import__("faa.data", fromlist=["x"]).load_table("digits", "target")
Xg = StandardScaler().fit_transform(Xg)
d = kdist(Xg, 10)
i = kneedle.knee(np.c_[np.arange(len(d)), d])
lab = DBSCAN(eps=float(d[i]), min_samples=10).fit_predict(Xg)
print(
    f"digits: knee eps {d[i]:.2f}, clusters {len(set(lab)) - (-1 in lab)}, noise {(lab == -1).mean():.2f}, ARI {ari(yg, lab):.3f}"
)

## Figures: minPts and dimension

The two sweeps above as curves.

In [ ]:
mps, ari_mp, cl_mp = [3, 5, 10, 20, 40], [], []
for m in mps:
    d = kdist(Xn, m)
    lab = DBSCAN(eps=float(d[kneedle.knee(np.c_[np.arange(len(d)), d])]), min_samples=m).fit_predict(Xn)
    ari_mp.append(ari(yn, lab)), cl_mp.append(len(set(lab)) - (-1 in lab))
f, ax = plt.subplots(figsize=(plotlib.COL, 1.5))
ax.plot(mps, ari_mp, "o-", color=BLUE, label="ARI")
ax.plot(mps, np.array(cl_mp) / max(cl_mp), "s--", color=ORANGE, label="clusters / max")
(
    ax.set_xscale("log"),
    ax.set_xticks(mps),
    ax.set_xticklabels(mps),
    ax.minorticks_off(),
    ax.set_xlabel("min_samples"),
    ax.legend(),
)
plotlib.save(f, "c1_minpts")
ds, ratio = [2, 10, 50, 200, 1000], []
for d_ in ds:
    Xd_ = np.random.default_rng(SEED).uniform(0, 1, (300, d_))
    D_ = np.linalg.norm(Xd_[:, None] - Xd_[None], axis=-1)
    np.fill_diagonal(D_, np.inf)
    ratio.append(float((np.where(np.isinf(D_), -np.inf, D_).max(1) / D_.min(1)).mean()))
f, ax = plt.subplots(figsize=(plotlib.COL, 1.5))
ax.plot(ds, ratio, "o-", color=RED)
ax.set_xscale("log"), ax.set_yscale("log"), ax.axhline(1, color=GREY, lw=0.6)
ax.set_xlabel("dimension $d$"), ax.set_ylabel("farthest / nearest")
plotlib.save(f, "c1_highdim")